### Ingesta de archivo "movie.csv"

In [0]:
%run "../../utilerias/funciones_comunes"

In [0]:
%run "../../utilerias/configurationes"

In [0]:
dbutils.widgets.text("param_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("param_file_date")


In [0]:
dbutils.widgets.text("parm_enviroment","")
v_enviroment = dbutils.widgets.get("parm_enviroment")

### Paso 1 - Leer el archivo CSV usando "DataframeReader" de Spark

In [0]:
#Importaremos las librerias necesarias para definir nuestros tipos de datos del nuevo esquema del dataframe
from pyspark.sql.types import IntegerType, StringType, DoubleType, DateType, StructType, StructField

In [0]:
movie_schema = StructType([
  StructField('movieId', IntegerType(), False),
  StructField('title', StringType(), True),
  StructField('budget', DoubleType(), True),
  StructField('homePage', StringType(), True),
  StructField('overview', StringType(), True),
  StructField('popularity', DoubleType(), True),
  StructField('yearReleaseDate', IntegerType(), True),
  StructField('releaseDate', DateType(), True),
  StructField('revenue', DoubleType(), True),
  StructField('durationTime', IntegerType(), True),
  StructField('movieStatus', StringType(), True),
  StructField('tagline', StringType(), True),
  StructField('voteAverage', DoubleType(), True),
  StructField('voteCount', IntegerType(), True)
])
#Creando el nuevo dataframe con el esquema definido 

In [0]:
movie_df = spark.read \
    .option('header', True)\
    .schema(movie_schema)\
.csv(f'{raw_folder_path}/{v_file_date}/movie.csv')

### 2 - Agregar columnas de auditoria al DataFrame.

In [0]:
#Importar la funcion
from pyspark.sql.functions import current_timestamp, lit

In [0]:
#Agregamos la nueva columna al dataframe
movie_df_final = movie_df.withColumns({"ingestion_date": current_timestamp()}).withColumns({"enviroment": lit(v_enviroment)}).withColumns({"file_date": lit(v_file_date) })

#### Escribir los datos en una tabla delta con external location.

In [0]:
merge_condition = 'tgt.movieId  = src.movieId  AND tgt.file_date = src.file_date'
merge_delta_lake("smartdata_proyect_bronze","movies",movie_df_final, merge_condition, "file_date")

In [0]:
%sql
select file_date, count(1) from smartdata_proyect_bronze.movies group by file_date;

In [0]:
%sql
select * from smartdata_proyect_bronze.movies  

In [0]:
dbutils.notebook.exit("success")